In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import NSGAIIISampler
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import os
import re
import subprocess
import time

C:\Users\hp\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# Paths
# ============================================================================
ECLRUN_EXE = r"C:\ecl\macros\eclrun.exe"
BASE_DIR = r"C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model"
DATA_FILE_NAME = "ECL_5SPOT.DATA"
SCHEDULE_FILE_NAME = os.path.join("include", "ECL_5SPOT_SCH.INC")
RSM_PATH = os.path.join(BASE_DIR, "ECL_5SPOT.RSM")

CASE_PATH = os.path.join(BASE_DIR, os.path.splitext(DATA_FILE_NAME)[0])   # ...\ECL_5SPOT
SCHED_PATH = os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)
SCHED_BACKUP = SCHED_PATH + ".orig"
RESULTS_CSV = os.path.join(BASE_DIR, "tpe_results.csv")

In [3]:
# ============================================================================
# Problem definition (paper)
# ============================================================================
INJECTORS = ["I1", "I2", "I3", "I4"]
PRODUCERS = ["P2", "P3"]
N_PERIODS = 6                       # BHP changed every 2 years, 6 times

INJ_BOUNDS = (220.0, 280.0)         # bar
PROD_BOUNDS = (90.0, 140.0)         # bar
BHP_STEP = None                     # None = continuous (paper: 279.4, 259.6, ...); e.g. 1.0 for integers

# Base case of the paper (Table 7), one value per period
BASE_CASE = {
    "I1": [258, 241, 251, 241, 257, 249],
    "I2": [220, 244, 273, 280, 266, 271],
    "I3": [222, 255, 235, 240, 266, 254],
    "I4": [224, 241, 280, 280, 237, 263],
    "P2": [113,  90, 133, 125, 130, 101],
    "P3": [140, 103, 135, 142, 145, 142],
}
# Note: P3 = 142 / 145 bar in the paper's base case is above the 140 bar bound.
# It is clipped to the bound so that it can be enqueued as a valid trial.

# Economics (paper, Table 6)
SM3_TO_BBL = 6.289811
OIL_PRICE = 50.0            # $/bbl
WATER_PROD_COST = 6.0       # $/bbl
WATER_INJ_COST = 2.0        # $/bbl
DISCOUNT_RATE = 0.02        # per interval
INTERVAL_DAYS = 91.25       # interval tau = 3 months
T_END = None                # days; None = end of the simulation (4380 d)

N_TRIALS = 300

# Display options
SHOW_ALL_TRIALS = True      # print the NPV of every evaluated case at the end
SORT_BY_NPV = False         # True: sort that table from best to worst NPV
TOP_N = None                # e.g. 20 to print only the 20 best cases (None = all)
PLOT_NPV = True             # plot NPV of each case + best-so-far curve (needs matplotlib)
PLOT_PATH = None            # None -> <BASE_DIR>\tpe_npv_history.png
N_STARTUP = 20
SEED = 42

In [6]:
SM3_TO_BBL = 6.289811

PAPER = {
    "oil_price": 50.0,          # $/bbl
    "water_prod_cost": 6.0,     # $/bbl (cost, entered positive)
    "water_inj_cost": 2.0,      # $/bbl (cost, entered positive)
    "discount_rate": 0.02,      # d, per interval tau
    "interval_days": 91.25,     # length of one interval tau (quarter)
    "t_start": 0.0,             # days, start of production
    "t_end": None,              # days, end of production (None = last summary time, 4380 d)
}


# ============================================================================
# 1. Read the summary
# ============================================================================
def _read_eclipse_binary(path):
    """Read an unformatted Eclipse file -> list of (keyword, values)."""
    with open(path, "rb") as f:
        data = f.read()
    pos, out = 0, []
    sizes = {"INTE": 4, "REAL": 4, "DOUB": 8, "LOGI": 4, "CHAR": 8}
    fmts = {"INTE": "i", "REAL": "f", "DOUB": "d", "LOGI": "i"}
    while pos < len(data):
        n = struct.unpack(">i", data[pos:pos + 4])[0]
        head = data[pos + 4:pos + 4 + n]
        pos += n + 8
        kw = head[:8].decode("ascii", "replace").strip()
        count = struct.unpack(">i", head[8:12])[0]
        typ = head[12:16].decode("ascii", "replace")
        size = sizes.get(typ, 4)
        vals, left = [], count
        while left > 0:
            n = struct.unpack(">i", data[pos:pos + 4])[0]
            blk = data[pos + 4:pos + 4 + n]
            pos += n + 8
            k = n // size
            if typ == "CHAR":
                vals += [blk[i * 8:(i + 1) * 8].decode("ascii", "replace").strip() for i in range(k)]
            else:
                vals += list(struct.unpack(">" + fmts.get(typ, "i") * k, blk))
            left -= k
        out.append((kw, vals))
    return out


def read_summary_binary(case_path, vectors=("FOPT", "FWPT", "FWIT")):
    """case_path: path without extension, e.g. r'C:\\...\\ECL_5SPOT'."""
    spec = dict(_read_eclipse_binary(case_path + ".SMSPEC"))
    keywords = spec["KEYWORDS"]
    units = spec.get("UNITS", [""] * len(keywords))
    col = {"TIME": keywords.index("TIME")}
    for v in vectors:
        if v not in keywords:
            raise KeyError(f"{v} is not in the SMSPEC file. Add it to the SUMMARY section.")
        col[v] = keywords.index(v)

    rows = {}
    for kw, vals in _read_eclipse_binary(case_path + ".UNSMRY"):
        if kw == "PARAMS":
            t = float(vals[col["TIME"]])
            rows[t] = {"time": t, **{v: float(vals[col[v]]) for v in vectors}}
    if units[col["TIME"]].upper().startswith("HOUR"):
        raise ValueError("TIME is not in days")
    return [rows[t] for t in sorted(rows)]


def _parse_multiplier(text):
    text = text.strip()
    return 10.0 ** int(text[5:]) if text.startswith("*10**") else 1.0


def read_summary(case_path):
    """Use SMSPEC/UNSMRY if present, otherwise the RSM file."""
    if os.path.exists(case_path + ".SMSPEC") and os.path.exists(case_path + ".UNSMRY"):
        return read_summary_binary(case_path)
    return read_summary_rsm(case_path + ".RSM")


# ============================================================================
# 2. NPV, Eq. (6)
# ============================================================================
def npv_paper(rows, oil_price=50.0, water_prod_cost=6.0, water_inj_cost=2.0,
              discount_rate=0.02, interval_days=91.25, t_start=0.0, t_end=None,
              details=False):
    """
    rows : list of {'time', 'FOPT', 'FWPT', 'FWIT'} (cumulative, SM3)

    The cumulative curves are evaluated at the interval boundaries
    t_start, t_start + interval_days, ..., t_end (linear interpolation between
    report steps = constant rates inside a report step). C_w,tau is the
    difference between two boundaries, and interval tau is discounted by
    (1 + d)^(tau - 1). A last interval shorter than interval_days is kept.
    """
    t = np.array([r["time"] for r in rows])
    t = np.concatenate(([0.0], t)) if t[0] > 0 else t
    def cum(v):
        y = np.array([r[v] for r in rows])
        return np.concatenate(([0.0], y)) if len(y) < len(t) else y
    Np, Wp, Wi = cum("FOPT"), cum("FWPT"), cum("FWIT")

    t_end = t[-1] if t_end is None else t_end
    edges = np.arange(t_start, t_end + 1e-9, interval_days)
    if t_end - edges[-1] > 1e-6:
        edges = np.append(edges, t_end)

    dNp = np.diff(np.interp(edges, t, Np)) * SM3_TO_BBL
    dWp = np.diff(np.interp(edges, t, Wp)) * SM3_TO_BBL
    dWi = np.diff(np.interp(edges, t, Wi)) * SM3_TO_BBL

    cash = oil_price * dNp - water_prod_cost * dWp - water_inj_cost * dWi
    tau = np.arange(1, len(cash) + 1)
    disc_cash = cash / (1.0 + discount_rate) ** (tau - 1)
    npv = float(disc_cash.sum())

    if details:
        table = list(zip(tau, edges[:-1], edges[1:], dNp, dWp, dWi, cash, disc_cash))
        return npv, table
    return npv


def print_table(npv, table):
    print(f"{'tau':>4} {'t0':>8} {'t1':>8} {'Oil (bbl)':>12} {'Wp (bbl)':>12} {'Wi (bbl)':>12} "
          f"{'Cash ($)':>14} {'Disc. ($)':>14}")
    for tau, t0, t1, dNp, dWp, dWi, c, dc in table:
        print(f"{tau:4d} {t0:8.2f} {t1:8.2f} {dNp:12,.0f} {dWp:12,.0f} {dWi:12,.0f} {c:14,.0f} {dc:14,.0f}")
    print(f"\nNPV = {npv:,.0f} USD  ({npv / 1e8:.4f} x 10^8 USD)")


# ============================================================================
# 3. Test
# ============================================================================
if __name__ == "__main__":
    BASE_DIR = r"C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model"
    CASE = os.path.join(BASE_DIR, "ECL_5SPOT")       # no extension

    rows = read_summary(CASE)
    last = rows[-1]
    print(f"Summary read: {len(rows)} time steps, last t = {last['time']} d")
    print(f"Np = {last['FOPT']/1e6:.3f} | Wi = {last['FWIT']/1e6:.3f} | Wp = {last['FWPT']/1e6:.3f}  (10^6 sm3)\n")

    npv, table = npv_paper(rows, **PAPER, details=True)
    print_table(npv, table)

    print("\nSensitivity to the discount convention:")
    for label, d, dt in [("no discount", 0.0, 91.25),
                         ("2 % per quarter (paper)", 0.02, 91.25),
                         ("2 %/yr compounded quarterly", 0.02 / 4, 91.25),
                         ("2 % per half year", 0.02, 182.5),
                         ("2 % per year", 0.02, 365.0)]:
        v = npv_paper(rows, **{**PAPER, "discount_rate": d, "interval_days": dt})
        print(f"  {label:30s}: {v / 1e8:.4f} x 10^8 USD")

Summary read: 67 time steps, last t = 4380.0 d
Np = 3.264 | Wi = 14.495 | Wp = 8.611  (10^6 sm3)

 tau       t0       t1    Oil (bbl)     Wp (bbl)     Wi (bbl)       Cash ($)      Disc. ($)
   1     0.00    91.25    1,384,915          163    1,826,599     65,591,580     65,591,580
   2    91.25   182.50    1,105,800          134    2,004,291     51,280,621     50,275,118
   3   182.50   273.75    1,123,468          129    2,065,997     52,040,624     50,019,823
   4   273.75   365.00    1,138,304          126    2,089,023     52,736,386     49,694,674
   5   365.00   456.25    1,146,429          122    2,105,411     53,109,909     49,065,346
   6   456.25   547.50    1,139,367        7,882    2,113,609     52,693,862     47,726,455
   7   547.50   638.75    1,039,051      181,386    2,111,405     46,641,398     41,416,226
   8   638.75   730.00      954,569      339,442    2,109,435     41,472,916     36,104,669
   9   730.00   821.25      701,993      399,912    1,899,516     28,901,1

# Optimize using TPE

In [18]:
# ============================================================================
# 0. Settings  (edit these)
# ============================================================================
BASE_DIR = r"C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model"
CASE_NAME = "ECL_5SPOT"
CASE = os.path.join(BASE_DIR, CASE_NAME)                      # no extension
SCH_PATH = os.path.join(BASE_DIR, "include", "ECL_5SPOT_SCH.INC")

# Command used to run Eclipse (same launcher as your current runs: eclrun).
# Add a version if needed, e.g. ["eclrun", "-v", "2023.1", "eclipse", ...]
ECLIPSE_CMD = ["eclrun", "eclipse", CASE_NAME + ".DATA"]
RUN_TIMEOUT_S = 1800

INJECTORS = ["I1", "I2", "I3", "I4"]        # injector 1..4
PRODUCERS = ["P2", "P3"]                    # producer 1..2
N_STEPS = 6                                 # control steps of 730 d
T_END = 4380.0                              # days, the run must reach this time

INJ_RANGE = (220.0, 280.0)                  # bar
PROD_RANGE = (90.0, 140.0)                  # bar
BHP_STEP = 0.01                             # resolution of the search (use 1 for integers)

N_TRIALS = 1200
SHOW_PROGRESS_BAR = False                   # Optuna progress bar
WRITE_BEST_TO_SCHEDULE = True               # at the end, leave the best BHPs in SCH_PATH
N_STARTUP = 20
SEED = 42
STORAGE = "sqlite:///" + os.path.join(BASE_DIR, "bhp_npv_optuna.db").replace("\\", "/")
STUDY_NAME = "ECL_5SPOT_BHP_NPV"
CSV_OUT = os.path.join(BASE_DIR, "bhp_npv_trials.csv")

SM3_TO_BBL = 6.289811
PAPER = {
    "oil_price": 50.0,          # $/bbl
    "water_prod_cost": 6.0,     # $/bbl
    "water_inj_cost": 2.0,      # $/bbl
    "discount_rate": 0.02,      # per interval
    "interval_days": 91.25,     # quarter
    "t_start": 0.0,
    "t_end": None,
}


# ----------------------------------------------------------------------------
# Variable names
# ----------------------------------------------------------------------------
def var_name(kind, well_idx, step):
    """kind 'I' or 'P', well_idx 1-based, step 1-based -> 'BHP-P11'."""
    return f"BHP-{kind}{well_idx}{step}"


def well_of(kind, well_idx):
    return (INJECTORS if kind == "I" else PRODUCERS)[well_idx - 1]


ALL_VARS = ([(var_name("I", i, s), "I", i, s) for s in range(1, N_STEPS + 1)
             for i in range(1, len(INJECTORS) + 1)] +
            [(var_name("P", p, s), "P", p, s) for s in range(1, N_STEPS + 1)
             for p in range(1, len(PRODUCERS) + 1)])


# ============================================================================
# 1. Read the summary (SMSPEC + UNSMRY)
# ============================================================================
def _read_eclipse_binary(path):
    with open(path, "rb") as f:
        data = f.read()
    pos, out = 0, []
    sizes = {"INTE": 4, "REAL": 4, "DOUB": 8, "LOGI": 4, "CHAR": 8}
    fmts = {"INTE": "i", "REAL": "f", "DOUB": "d", "LOGI": "i"}
    while pos < len(data):
        n = struct.unpack(">i", data[pos:pos + 4])[0]
        head = data[pos + 4:pos + 4 + n]
        pos += n + 8
        kw = head[:8].decode("ascii", "replace").strip()
        count = struct.unpack(">i", head[8:12])[0]
        typ = head[12:16].decode("ascii", "replace")
        size = sizes.get(typ, 4)
        vals, left = [], count
        while left > 0:
            n = struct.unpack(">i", data[pos:pos + 4])[0]
            blk = data[pos + 4:pos + 4 + n]
            pos += n + 8
            k = n // size
            if typ == "CHAR":
                vals += [blk[i * 8:(i + 1) * 8].decode("ascii", "replace").strip() for i in range(k)]
            else:
                vals += list(struct.unpack(">" + fmts.get(typ, "i") * k, blk))
            left -= k
        out.append((kw, vals))
    return out


def read_summary_binary(case_path, vectors=("FOPT", "FWPT", "FWIT")):
    spec = dict(_read_eclipse_binary(case_path + ".SMSPEC"))
    keywords = spec["KEYWORDS"]
    col = {"TIME": keywords.index("TIME")}
    for v in vectors:
        if v not in keywords:
            raise KeyError(f"{v} is not in the SMSPEC file. Add it to the SUMMARY section.")
        col[v] = keywords.index(v)
    rows = {}
    for kw, vals in _read_eclipse_binary(case_path + ".UNSMRY"):
        if kw == "PARAMS":
            t = float(vals[col["TIME"]])
            rows[t] = {"time": t, **{v: float(vals[col[v]]) for v in vectors}}
    return [rows[t] for t in sorted(rows)]


# ============================================================================
# 2. NPV, Eq. (6)
# ============================================================================
def npv_paper(rows, oil_price=50.0, water_prod_cost=6.0, water_inj_cost=2.0,
              discount_rate=0.02, interval_days=91.25, t_start=0.0, t_end=None):
    t = np.array([r["time"] for r in rows])
    t = np.concatenate(([0.0], t)) if t[0] > 0 else t

    def cum(v):
        y = np.array([r[v] for r in rows])
        return np.concatenate(([0.0], y)) if len(y) < len(t) else y
    Np, Wp, Wi = cum("FOPT"), cum("FWPT"), cum("FWIT")

    t_end = t[-1] if t_end is None else t_end
    edges = np.arange(t_start, t_end + 1e-9, interval_days)
    if t_end - edges[-1] > 1e-6:
        edges = np.append(edges, t_end)

    dNp = np.diff(np.interp(edges, t, Np)) * SM3_TO_BBL
    dWp = np.diff(np.interp(edges, t, Wp)) * SM3_TO_BBL
    dWi = np.diff(np.interp(edges, t, Wi)) * SM3_TO_BBL
    cash = oil_price * dNp - water_prod_cost * dWp - water_inj_cost * dWi
    tau = np.arange(1, len(cash) + 1)
    return float((cash / (1.0 + discount_rate) ** (tau - 1)).sum())


# ============================================================================
# 3. Modify the schedule file (in place)
# ============================================================================
# A well record inside WCONINJE / WCONPROD, e.g.
#    I1 WATER OPEN BHP 1* 1* 258/
#    P2 OPEN BHP 1* 1* 1* 1* 1* 113/
# The last number before "/" is the BHP target.
_REC = re.compile(r"^(?P<pre>\s*'?(?P<well>[A-Za-z0-9_]+)'?\s.*?\s)"
                  r"(?P<bhp>[0-9]+(?:\.[0-9]*)?(?:[eE][-+]?[0-9]+)?)"
                  r"(?P<post>\s*/.*)$")


def _read_lines(path):
    """Read the file, keep its line ending (CRLF on Windows files)."""
    with open(path, "r", newline="") as f:
        text = f.read()
    eol = "\r\n" if "\r\n" in text else "\n"
    return text.splitlines(), eol


def _scan_schedule(lines):
    """Return {(well, step): line_index} for every BHP record of the schedule.
    step = 1, 2, ... = order in which the well appears in WCONINJE/WCONPROD."""
    loc, count, kw = {}, {}, None
    for i, ln in enumerate(lines):
        s = ln.strip()
        if not s or s.startswith("--"):
            continue
        if re.match(r"^[A-Z][A-Z0-9_]{0,7}$", s):        # keyword line
            kw = s
            continue
        if s.startswith("/"):                             # end of keyword
            kw = None
            continue
        if kw in ("WCONINJE", "WCONPROD"):
            m = _REC.match(ln)
            if m and m.group("well") in INJECTORS + PRODUCERS:
                w = m.group("well")
                count[w] = count.get(w, 0) + 1
                loc[(w, count[w])] = i
    missing = [(well_of(k, i), st) for _, k, i, st in ALL_VARS
               if (well_of(k, i), st) not in loc]
    if missing:
        raise RuntimeError(f"No WCONINJE/WCONPROD record found in {SCH_PATH} for {missing}")
    return loc


def read_schedule_bhp(path=SCH_PATH):
    """BHPs currently written in the schedule, as {var_name: value}."""
    lines, _ = _read_lines(path)
    loc = _scan_schedule(lines)
    return {name: float(_REC.match(lines[loc[(well_of(k, i), st)]]).group("bhp"))
            for name, k, i, st in ALL_VARS}


def _fmt(v):
    return f"{int(round(v))}" if float(BHP_STEP).is_integer() else f"{v:.2f}"


def modify_files(params, path=SCH_PATH):
    """Write the BHPs of `params` directly into the schedule file, then check them."""
    lines, eol = _read_lines(path)
    loc = _scan_schedule(lines)
    for name, kind, idx, step in ALL_VARS:
        j = loc[(well_of(kind, idx), step)]
        m = _REC.match(lines[j])
        lines[j] = m.group("pre") + _fmt(params[name]) + m.group("post")
    with open(path, "w", newline="") as f:
        f.write(eol.join(lines) + eol)

    # make sure the file really contains the new values
    written = read_schedule_bhp(path)
    bad = {n: (written[n], params[n]) for n in params
           if abs(written[n] - float(_fmt(params[n]))) > 1e-6}
    if bad:
        raise RuntimeError(f"Schedule not modified correctly: {bad}")


# ============================================================================
# 4. Run Eclipse
# ============================================================================
SUMMARY_EXT = (".SMSPEC", ".UNSMRY", ".RSM", ".PRT")


def run_eclipse():
    """Run the deck; return True if the run reached T_END without errors."""
    for ext in SUMMARY_EXT:                      # never read results of an old run
        p = CASE + ext
        if os.path.exists(p):
            os.remove(p)
    try:
        subprocess.run(ECLIPSE_CMD, cwd=BASE_DIR, timeout=RUN_TIMEOUT_S,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                       shell=(os.name == "nt"))
    except subprocess.TimeoutExpired:
        return False
    # eclrun may return 1 even when the run is fine (affinity message), so the
    # result files are checked instead of the return code
    if not (os.path.exists(CASE + ".SMSPEC") and os.path.exists(CASE + ".UNSMRY")):
        return False
    prt = CASE + ".PRT"
    if os.path.exists(prt):
        with open(prt, errors="ignore") as f:
            txt = f.read()
        m = re.search(r"Errors\s+(\d+)", txt)
        if m and int(m.group(1)) > 0:
            return False
    return True


# ============================================================================
# 5. Optuna objective
# ============================================================================
def suggest(trial, name, low, high):
    if float(BHP_STEP).is_integer():
        return trial.suggest_int(name, int(low), int(high), step=int(BHP_STEP))
    return trial.suggest_float(name, low, high, step=BHP_STEP)


def objective(trial):
    # 1. TPE parameter suggestions
    params = {}
    for name, kind, idx, step in ALL_VARS:
        lo, hi = INJ_RANGE if kind == "I" else PROD_RANGE
        params[name] = suggest(trial, name, lo, hi)

    # 2. Modify Eclipse input file
    modify_files(params)

    # 3. Run Eclipse
    ok = run_eclipse()
    if not ok:
        trial.set_user_attr("status", "eclipse_failed")
        raise optuna.TrialPruned("Eclipse run failed or did not finish")

    # 4. Read results
    rows = read_summary_binary(CASE)
    if rows[-1]["time"] < T_END - 1e-3:
        trial.set_user_attr("status", f"stopped_at_{rows[-1]['time']:.1f}d")
        raise optuna.TrialPruned("Run stopped before the end time")

    # 5. NPV
    npv = npv_paper(rows, **PAPER)

    last = rows[-1]
    trial.set_user_attr("FOPT", last["FOPT"])
    trial.set_user_attr("FWPT", last["FWPT"])
    trial.set_user_attr("FWIT", last["FWIT"])
    trial.set_user_attr("NPV", npv)
    trial.set_user_attr("status", "ok")
    print(f"Trial {trial.number:4d}: NPV = {npv / 1e8:.4f} x 10^8 USD")

    # 6. Objective
    return npv


# ============================================================================
# 6. Main
# ============================================================================
def export_csv(study):
    names = [v[0] for v in ALL_VARS]
    with open(CSV_OUT, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["trial", "state", "NPV_USD", "FOPT_sm3", "FWPT_sm3", "FWIT_sm3"] + names)
        for t in study.trials:
            ua = t.user_attrs
            w.writerow([t.number, t.state.name, ua.get("NPV"), ua.get("FOPT"),
                        ua.get("FWPT"), ua.get("FWIT")] + [t.params.get(n) for n in names])


def print_schedule(params):
    head = "step  " + "  ".join(f"{w:>7s}" for w in INJECTORS + PRODUCERS)
    print(head)
    for s in range(1, N_STEPS + 1):
        vals = [params[var_name("I", i, s)] for i in range(1, len(INJECTORS) + 1)] + \
               [params[var_name("P", p, s)] for p in range(1, len(PRODUCERS) + 1)]
        print(f"{s:4d}  " + "  ".join(f"{v:7.2f}" for v in vals))


def test_modify():
    """python optimize_bhp_npv.py --test-modify : write random BHPs, show the file changed."""
    import random
    before = read_bhp(SCH_PATH)
    rnd = {}
    for name, kind, idx, step in ALL_VARS:
        lo, hi = INJ_RANGE if kind == "I" else PROD_RANGE
        rnd[name] = random.randint(int(lo), int(hi))
    modify_files(rnd)
    after = read_bhp(SCH_PATH)
    print("Before:")
    print_schedule(before)
    print("After (read back from the file):")
    print_schedule(after)
    modify_files(before)                      # put the original values back
    print("\nOK: all 36 BHPs were written and read back. Original values restored.")


if __name__ == "__main__":
    # baseline = the BHPs currently in the schedule file (clipped to the search space)
    BASELINE_PARAMS = {}
    for name, val in read_schedule_bhp().items():
        lo, hi = INJ_RANGE if name.startswith("BHP-I") else PROD_RANGE
        v = min(max(val, lo), hi)
        BASELINE_PARAMS[name] = int(round(v)) if float(BHP_STEP).is_integer() else round(v, 2)
    print("Baseline schedule (bar):")
    print_schedule(BASELINE_PARAMS)

    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=STORAGE,              # results kept in a .db file -> run can be resumed
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP),
    )
    if len(study.trials) == 0:
        study.enqueue_trial(BASELINE_PARAMS)

    done = [t for t in study.trials if t.state != optuna.trial.TrialState.WAITING]
    remaining = N_TRIALS - len(done)
    try:
        if remaining > 0:
            study.optimize(objective, n_trials=remaining,
                           show_progress_bar=SHOW_PROGRESS_BAR)
    finally:
        export_csv(study)

    best = study.best_trial
    print(f"\nBest trial #{best.number}: NPV = {best.value:,.0f} USD ({best.value / 1e8:.4f} x 10^8)")
    print(f"FOPT = {best.user_attrs['FOPT']/1e6:.3f} | FWPT = {best.user_attrs['FWPT']/1e6:.3f}"
          f" | FWIT = {best.user_attrs['FWIT']/1e6:.3f}  (10^6 sm3)")
    print("Best BHP schedule (bar):")
    print_schedule(best.params)

    if WRITE_BEST_TO_SCHEDULE:
        modify_files(best.params)
        print(f"\nBest BHPs written into {SCH_PATH}")
    print(f"All trials written to {CSV_OUT}")

Baseline schedule (bar):
step       I1       I2       I3       I4       P2       P3
   1   220.00   280.00   279.60   223.20   140.00   140.00
   2   220.00   279.40   279.90   220.00   140.00   140.00
   3   220.00   280.00   279.90   220.00   140.00   140.00
   4   220.00   280.00   279.40   220.00   140.00   140.00
   5   220.00   279.90   259.60   220.00   140.00   140.00
   6   220.00   220.00   220.10   220.00   140.00   139.90


[I 2026-10-28 19:58:35,114] Using an existing study with name 'ECL_5SPOT_BHP_NPV' instead of creating a new one.
[I 2026-10-28 19:58:52,116] Trial 367 finished with value: 537429252.2994483 and parameters: {'BHP-I11': 223.38, 'BHP-I21': 277.83, 'BHP-I31': 273.12, 'BHP-I41': 268.97, 'BHP-I12': 226.99, 'BHP-I22': 260.91, 'BHP-I32': 271.09000000000003, 'BHP-I42': 231.34, 'BHP-I13': 224.96, 'BHP-I23': 222.84, 'BHP-I33': 224.24, 'BHP-I43': 261.02, 'BHP-I14': 222.55, 'BHP-I24': 237.19, 'BHP-I34': 243.44, 'BHP-I44': 244.42000000000002, 'BHP-I15': 231.02, 'BHP-I25': 277.39, 'BHP-I35': 235.6, 'BHP-I45': 233.06, 'BHP-I16': 231.47, 'BHP-I26': 226.57, 'BHP-I36': 225.05, 'BHP-I46': 259.54, 'BHP-P11': 93.74, 'BHP-P21': 93.86, 'BHP-P12': 108.34, 'BHP-P22': 125.6, 'BHP-P13': 138.62, 'BHP-P23': 105.32, 'BHP-P14': 123.53999999999999, 'BHP-P24': 136.69, 'BHP-P15': 123.05, 'BHP-P25': 139.99, 'BHP-P16': 119.25, 'BHP-P26': 134.48000000000002}. Best is trial 367 with value: 537429252.2994483.


Trial  367: NPV = 5.3743 x 10^8 USD


[I 2026-10-28 19:59:07,473] Trial 368 finished with value: 536365683.0653398 and parameters: {'BHP-I11': 221.67, 'BHP-I21': 276.12, 'BHP-I31': 274.33, 'BHP-I41': 269.22, 'BHP-I12': 227.3, 'BHP-I22': 260.39, 'BHP-I32': 271.18, 'BHP-I42': 230.93, 'BHP-I13': 226.19, 'BHP-I23': 222.71, 'BHP-I33': 223.54, 'BHP-I43': 261.05, 'BHP-I14': 224.65, 'BHP-I24': 236.5, 'BHP-I34': 243.83, 'BHP-I44': 244.83, 'BHP-I15': 231.25, 'BHP-I25': 277.2, 'BHP-I35': 239.12, 'BHP-I45': 233.5, 'BHP-I16': 234.14, 'BHP-I26': 226.82, 'BHP-I36': 226.03, 'BHP-I46': 259.12, 'BHP-P11': 95.82, 'BHP-P21': 94.11, 'BHP-P12': 108.65, 'BHP-P22': 124.87, 'BHP-P13': 139.15, 'BHP-P23': 105.23, 'BHP-P14': 123.32, 'BHP-P24': 136.64, 'BHP-P15': 124.7, 'BHP-P25': 138.3, 'BHP-P16': 117.77, 'BHP-P26': 134.65}. Best is trial 367 with value: 537429252.2994483.


Trial  368: NPV = 5.3637 x 10^8 USD


[I 2026-10-28 19:59:22,808] Trial 369 finished with value: 535784265.5473887 and parameters: {'BHP-I11': 221.38, 'BHP-I21': 276.06, 'BHP-I31': 274.4, 'BHP-I41': 268.99, 'BHP-I12': 227.73, 'BHP-I22': 259.78, 'BHP-I32': 267.9, 'BHP-I42': 231.0, 'BHP-I13': 226.26, 'BHP-I23': 222.82, 'BHP-I33': 222.8, 'BHP-I43': 259.31, 'BHP-I14': 224.94, 'BHP-I24': 235.97, 'BHP-I34': 246.52, 'BHP-I44': 244.81, 'BHP-I15': 232.99, 'BHP-I25': 274.62, 'BHP-I35': 238.54, 'BHP-I45': 235.39, 'BHP-I16': 234.33, 'BHP-I26': 226.69, 'BHP-I36': 224.56, 'BHP-I46': 259.02, 'BHP-P11': 95.45, 'BHP-P21': 94.09, 'BHP-P12': 109.83, 'BHP-P22': 125.53, 'BHP-P13': 139.16, 'BHP-P23': 105.47, 'BHP-P14': 124.07, 'BHP-P24': 135.04, 'BHP-P15': 125.03, 'BHP-P25': 138.37, 'BHP-P16': 117.91, 'BHP-P26': 132.66}. Best is trial 367 with value: 537429252.2994483.


... (1,651 outputs left out)

[I 2026-10-28 23:27:06,754] Trial 1198 finished with value: 541124374.1441396 and parameters: {'BHP-I11': 240.18, 'BHP-I21': 262.37, 'BHP-I31': 246.38, 'BHP-I41': 264.14, 'BHP-I12': 227.94, 'BHP-I22': 262.42, 'BHP-I32': 263.47, 'BHP-I42': 243.9, 'BHP-I13': 228.82, 'BHP-I23': 277.94, 'BHP-I33': 276.67, 'BHP-I43': 234.42, 'BHP-I14': 236.99, 'BHP-I24': 220.99, 'BHP-I34': 257.07, 'BHP-I44': 221.22, 'BHP-I15': 228.5, 'BHP-I25': 221.25, 'BHP-I35': 270.27, 'BHP-I45': 232.4, 'BHP-I16': 257.09000000000003, 'BHP-I26': 274.8, 'BHP-I36': 248.13, 'BHP-I46': 240.07, 'BHP-P11': 92.28, 'BHP-P21': 107.96000000000001, 'BHP-P12': 136.11, 'BHP-P22': 100.12, 'BHP-P13': 136.14, 'BHP-P23': 139.87, 'BHP-P14': 136.64, 'BHP-P24': 133.96, 'BHP-P15': 139.93, 'BHP-P25': 139.11, 'BHP-P16': 129.15, 'BHP-P26': 133.42000000000002}. Best is trial 1141 with value: 556653271.9754117.


Trial 1198: NPV = 5.4112 x 10^8 USD


[I 2026-10-28 23:27:23,843] Trial 1199 finished with value: 540829341.5667703 and parameters: {'BHP-I11': 239.29, 'BHP-I21': 264.39, 'BHP-I31': 265.15, 'BHP-I41': 265.07, 'BHP-I12': 229.17, 'BHP-I22': 262.85, 'BHP-I32': 263.15, 'BHP-I42': 245.26, 'BHP-I13': 229.4, 'BHP-I23': 276.57, 'BHP-I33': 279.93, 'BHP-I43': 233.98, 'BHP-I14': 235.6, 'BHP-I24': 220.95, 'BHP-I34': 258.65, 'BHP-I44': 221.14, 'BHP-I15': 229.06, 'BHP-I25': 222.6, 'BHP-I35': 269.58, 'BHP-I45': 231.91, 'BHP-I16': 279.85, 'BHP-I26': 273.96, 'BHP-I36': 248.4, 'BHP-I46': 240.82, 'BHP-P11': 92.63, 'BHP-P21': 107.44, 'BHP-P12': 136.5, 'BHP-P22': 101.1, 'BHP-P13': 136.07, 'BHP-P23': 139.09, 'BHP-P14': 138.17000000000002, 'BHP-P24': 135.35, 'BHP-P15': 139.99, 'BHP-P25': 139.15, 'BHP-P16': 130.69, 'BHP-P26': 132.43}. Best is trial 1141 with value: 556653271.9754117.


Trial 1199: NPV = 5.4083 x 10^8 USD

Best trial #1141: NPV = 556,653,272 USD (5.5665 x 10^8)
FOPT = 3.212 | FWPT = 6.690 | FWIT = 12.494  (10^6 sm3)
Best BHP schedule (bar):
step       I1       I2       I3       I4       P2       P3
   1   242.39   266.27   263.22   265.01    91.09   107.04
   2   226.90   264.52   262.86   246.14   136.66    96.23
   3   229.84   278.64   278.71   230.54   135.18   138.26
   4   234.66   223.04   261.22   222.76   137.79   130.04
   5   226.69   221.82   276.07   228.60   138.04   139.34
   6   226.37   273.79   253.09   241.08   131.28   132.87

Best BHPs written into C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model\include\ECL_5SPOT_SCH.INC
All trials written to C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model\bhp_npv_trials.csv


# Alfarizi optimal case running NPV

In [17]:
# ============================================================================
# 0. Settings
# ============================================================================
BASE_DIR = r"C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model"
CASE_NAME = "ECL_5SPOT"
CASE = os.path.join(BASE_DIR, CASE_NAME)
SCH_PATH = os.path.join(BASE_DIR, "include", "ECL_5SPOT_SCH.INC")
ECLIPSE_CMD = ["eclrun", "eclipse", CASE_NAME + ".DATA"]
RUN_ECLIPSE = True          # False = only compute the NPV of the results already in the folder

WELLS = ["I1", "I2", "I3", "I4", "P2", "P3"]

# Paper optimal case (bar).  <-- replace with the paper's table if different
#   step      I1      I2      I3      I4      P2      P3
PAPER_OPTIMAL_BHP = [
    [220, 280, 279.6, 223.2, 140, 140],   # step 1:    0 -  730 d
    [220, 279.4, 279.9, 220, 140, 140],   # step 2:  730 - 1460 d
    [220, 280, 279.9, 220, 140,  140],   # step 3: 1460 - 2190 d
    [220, 280, 279.4, 220, 140,  140],   # step 4: 2190 - 2920 d
    [220, 279.9, 259.6, 220, 140,  140],   # step 5: 2920 - 3650 d
    [220, 220, 220.1, 220, 140, 139.9],   # step 6: 3650 - 4380 d
]

PAPER_NPV = None            # paper's NPV in USD, e.g. 5.13e8, to print the difference

SM3_TO_BBL = 6.289811
PAPER = {
    "oil_price": 50.0,          # $/bbl
    "water_prod_cost": 6.0,     # $/bbl
    "water_inj_cost": 2.0,      # $/bbl
    "discount_rate": 0.02,      # per interval
    "interval_days": 91.25,     # quarter
    "t_start": 0.0,
    "t_end": None,
}


# ============================================================================
# 1. Write the BHPs into the schedule
# ============================================================================
_REC = re.compile(r"^(?P<pre>\s*'?(?P<well>[A-Za-z0-9_]+)'?\s.*?\s)"
                  r"(?P<bhp>[0-9]+(?:\.[0-9]*)?)(?P<post>\s*/.*)$")


def write_schedule(table, path=SCH_PATH):
    with open(path, "r", newline="") as f:
        text = f.read()
    eol = "\r\n" if "\r\n" in text else "\n"
    lines = text.splitlines()

    count, kw, done = {}, None, 0
    for i, ln in enumerate(lines):
        s = ln.strip()
        if not s or s.startswith("--"):
            continue
        if re.match(r"^[A-Z][A-Z0-9_]{0,7}$", s):
            kw = s
            continue
        if s.startswith("/"):
            kw = None
            continue
        if kw in ("WCONINJE", "WCONPROD"):
            m = _REC.match(ln)
            if m and m.group("well") in WELLS:
                w = m.group("well")
                count[w] = count.get(w, 0) + 1
                step = count[w]
                if step > len(table):
                    raise RuntimeError(f"{w} has more control steps in the schedule than in the table")
                v = table[step - 1][WELLS.index(w)]
                lines[i] = m.group("pre") + f"{v:.2f}" + m.group("post")
                done += 1

    expected = len(table) * len(WELLS)
    if done != expected:
        raise RuntimeError(f"Changed {done} BHP records, expected {expected}: {count}")
    with open(path, "w", newline="") as f:
        f.write(eol.join(lines) + eol)
    print(f"{done} BHP values written into {path}")


# ============================================================================
# 2. Run Eclipse
# ============================================================================
def run_eclipse():
    for ext in (".SMSPEC", ".UNSMRY", ".RSM", ".PRT"):
        if os.path.exists(CASE + ext):
            os.remove(CASE + ext)
    print("Running Eclipse ...")
    subprocess.run(ECLIPSE_CMD, cwd=BASE_DIR, shell=(os.name == "nt"))
    if not (os.path.exists(CASE + ".SMSPEC") and os.path.exists(CASE + ".UNSMRY")):
        raise RuntimeError("Eclipse did not write SMSPEC/UNSMRY - check the PRT/LOG file")
    m = None
    if os.path.exists(CASE + ".PRT"):
        with open(CASE + ".PRT", errors="ignore") as f:
            m = re.search(r"Errors\s+(\d+)", f.read())
    if m and int(m.group(1)) > 0:
        raise RuntimeError(f"Eclipse reported {m.group(1)} errors - check the PRT file")


# ============================================================================
# 3. Read the summary
# ============================================================================
def _read_eclipse_binary(path):
    with open(path, "rb") as f:
        data = f.read()
    pos, out = 0, []
    sizes = {"INTE": 4, "REAL": 4, "DOUB": 8, "LOGI": 4, "CHAR": 8}
    fmts = {"INTE": "i", "REAL": "f", "DOUB": "d", "LOGI": "i"}
    while pos < len(data):
        n = struct.unpack(">i", data[pos:pos + 4])[0]
        head = data[pos + 4:pos + 4 + n]
        pos += n + 8
        kw = head[:8].decode("ascii", "replace").strip()
        count = struct.unpack(">i", head[8:12])[0]
        typ = head[12:16].decode("ascii", "replace")
        size = sizes.get(typ, 4)
        vals, left = [], count
        while left > 0:
            n = struct.unpack(">i", data[pos:pos + 4])[0]
            blk = data[pos + 4:pos + 4 + n]
            pos += n + 8
            k = n // size
            if typ == "CHAR":
                vals += [blk[i * 8:(i + 1) * 8].decode("ascii", "replace").strip() for i in range(k)]
            else:
                vals += list(struct.unpack(">" + fmts.get(typ, "i") * k, blk))
            left -= k
        out.append((kw, vals))
    return out


def read_summary_binary(case_path, vectors=("FOPT", "FWPT", "FWIT")):
    spec = dict(_read_eclipse_binary(case_path + ".SMSPEC"))
    keywords = spec["KEYWORDS"]
    col = {"TIME": keywords.index("TIME")}
    for v in vectors:
        if v not in keywords:
            raise KeyError(f"{v} is not in the SMSPEC file. Add it to the SUMMARY section.")
        col[v] = keywords.index(v)
    rows = {}
    for kw, vals in _read_eclipse_binary(case_path + ".UNSMRY"):
        if kw == "PARAMS":
            t = float(vals[col["TIME"]])
            rows[t] = {"time": t, **{v: float(vals[col[v]]) for v in vectors}}
    return [rows[t] for t in sorted(rows)]


# ============================================================================
# 4. NPV, Eq. (6)
# ============================================================================
def npv_paper(rows, oil_price=50.0, water_prod_cost=6.0, water_inj_cost=2.0,
              discount_rate=0.02, interval_days=91.25, t_start=0.0, t_end=None,
              details=False):
    t = np.array([r["time"] for r in rows])
    t = np.concatenate(([0.0], t)) if t[0] > 0 else t

    def cum(v):
        y = np.array([r[v] for r in rows])
        return np.concatenate(([0.0], y)) if len(y) < len(t) else y
    Np, Wp, Wi = cum("FOPT"), cum("FWPT"), cum("FWIT")

    t_end = t[-1] if t_end is None else t_end
    edges = np.arange(t_start, t_end + 1e-9, interval_days)
    if t_end - edges[-1] > 1e-6:
        edges = np.append(edges, t_end)

    dNp = np.diff(np.interp(edges, t, Np)) * SM3_TO_BBL
    dWp = np.diff(np.interp(edges, t, Wp)) * SM3_TO_BBL
    dWi = np.diff(np.interp(edges, t, Wi)) * SM3_TO_BBL
    cash = oil_price * dNp - water_prod_cost * dWp - water_inj_cost * dWi
    tau = np.arange(1, len(cash) + 1)
    disc_cash = cash / (1.0 + discount_rate) ** (tau - 1)
    npv = float(disc_cash.sum())
    if details:
        return npv, list(zip(tau, edges[:-1], edges[1:], dNp, dWp, dWi, cash, disc_cash))
    return npv


def print_table(npv, table):
    print(f"{'tau':>4} {'t0':>8} {'t1':>8} {'Oil (bbl)':>12} {'Wp (bbl)':>12} {'Wi (bbl)':>12} "
          f"{'Cash ($)':>14} {'Disc. ($)':>14}")
    for tau, t0, t1, dNp, dWp, dWi, c, dc in table:
        print(f"{tau:4d} {t0:8.2f} {t1:8.2f} {dNp:12,.0f} {dWp:12,.0f} {dWi:12,.0f} {c:14,.0f} {dc:14,.0f}")
    print(f"\nNPV = {npv:,.0f} USD  ({npv / 1e8:.4f} x 10^8 USD)")


# ============================================================================
# 5. Main
# ============================================================================
if __name__ == "__main__":
    print("Paper optimal BHP schedule (bar):")
    print("step  " + "  ".join(f"{w:>7s}" for w in WELLS))
    for s, row in enumerate(PAPER_OPTIMAL_BHP, 1):
        print(f"{s:4d}  " + "  ".join(f"{v:7.2f}" for v in row))
    print()

    if RUN_ECLIPSE:
        write_schedule(PAPER_OPTIMAL_BHP)
        run_eclipse()

    rows = read_summary_binary(CASE)
    last = rows[-1]
    print(f"\nSummary read: {len(rows)} time steps, last t = {last['time']} d")
    print(f"Np = {last['FOPT']/1e6:.3f} | Wi = {last['FWIT']/1e6:.3f} | Wp = {last['FWPT']/1e6:.3f}  (10^6 sm3)\n")

    npv, table = npv_paper(rows, **PAPER, details=True)
    print_table(npv, table)

    if PAPER_NPV:
        diff = npv - PAPER_NPV
        print(f"Paper NPV = {PAPER_NPV / 1e8:.4f} x 10^8 USD  ->  difference = "
              f"{diff / 1e6:+.2f} x 10^6 USD ({100 * diff / PAPER_NPV:+.2f} %)")

Paper optimal BHP schedule (bar):
step       I1       I2       I3       I4       P2       P3
   1   220.00   280.00   279.60   223.20   140.00   140.00
   2   220.00   279.40   279.90   220.00   140.00   140.00
   3   220.00   280.00   279.90   220.00   140.00   140.00
   4   220.00   280.00   279.40   220.00   140.00   140.00
   5   220.00   279.90   259.60   220.00   140.00   140.00
   6   220.00   220.00   220.10   220.00   140.00   139.90

36 BHP values written into C:\Users\hp\Desktop\Muhammad Gibran Alfarizi Model\include\ECL_5SPOT_SCH.INC
Running Eclipse ...

Summary read: 71 time steps, last t = 4380.0 d
Np = 3.137 | Wi = 10.552 | Wp = 4.890  (10^6 sm3)

 tau       t0       t1    Oil (bbl)     Wp (bbl)     Wi (bbl)       Cash ($)      Disc. ($)
   1     0.00    91.25      777,067           47    1,321,001     36,211,062     36,211,062
   2    91.25   182.50      712,752           44    1,328,969     32,979,412     32,332,757
   3   182.50   273.75      734,310           43    1